# HeadScan-Lite — export existing trained model to ONNX

**Use this notebook after training.** It mounts Google Drive, installs the latest `Head_model`, reads the existing best checkpoint, and exports/validates ONNX. **No dataset restore, synthesis, or retraining occurs.**

Reads: `MyDrive/head_model/headscan-lite-makehuman-2500/best.pt`.

Writes: `MyDrive/head_model/headscan-lite-makehuman-2500/headscan_lite.onnx`.

Uses the stable TorchScript ONNX exporter (opset 17), disables fused attention during export, and checks ONNX Runtime parity. Errors show their complete traceback directly.


In [ ]:
from pathlib import Path
import subprocess, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)
repo = Path('/content/headscan_lite/Head_model')
run_dir = Path('/content/drive/MyDrive/head_model/headscan-lite-makehuman-2500')
best = run_dir / 'best.pt'
onnx_path = run_dir / 'headscan_lite.onnx'
if not best.is_file():
    raise FileNotFoundError(f'Trained best.pt checkpoint not found: {best}')
repo.parent.mkdir(parents=True, exist_ok=True)
if (repo / '.git').is_dir():
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Persie0/Head_model.git', str(repo)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', f'{repo}[export]'], check=True)
print(f'[export-only] Using existing checkpoint: {best}', flush=True)
subprocess.run([sys.executable, '-u', '-m', 'head_model.export', '--checkpoint', str(best), '--output', str(onnx_path)], cwd=str(repo), check=True)
print(f'[export-only] ONNX ready: {onnx_path} ({onnx_path.stat().st_size/1024/1024:.1f} MiB)', flush=True)
